# Bioyologic · Clasificador de especie por colonia (Google Colab)

Segunda etapa de Bioyologic: el **detector** ya encuentra y cuenta las colonias; este notebook entrena un **clasificador** (YOLO11-cls) que mira el recorte de cada colonia y dice su especie. La especie de la placa se decide por votación entre sus colonias.

Datos: los mismos dos datasets públicos (Makrai *et al.* 2023 y el dataset clínico de Du, 2026, publicado en Zenodo), unificados por nombre de especie: **37 especies**.

**Cómo usarlo**: `Entorno de ejecución → Cambiar tipo → GPU T4` y `Ejecutar todo` (~1,5 h). Al final se descarga `bioyologic_species.zip` (modelo + configuración): descomprímelo en `models/`.

**Advertencia**: en estos datasets cada placa tiene una sola especie y cada especie se cultivó en su propio medio (sangre, MacConkey, etc.). El clasificador puede apoyarse en el color del agar además de en la colonia. Con fotos de otro laboratorio, la especie es orientativa y debe confirmarse con pruebas microbiológicas.

**Añadir más datos (opcional).** En la celda 2, `EXTRA` acepta carpetas o `.zip` con placas de especie conocida (por ejemplo, las de tu laboratorio exportadas con `python -m bioyologic.manage export-coco --especie "…"`). Es la forma de reducir el sesgo del medio de cultivo: las mismas especies en otros agares, con otras cámaras. Las placas de test de los dos datasets públicos **no cambian**: la celda 8 mide el clasificador nuevo y el actual sobre las mismas placas y dice si conviene reemplazarlo.


In [ ]:
# 1. GPU y dependencias
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Sin GPU: activa T4 en 'Cambiar tipo de entorno de ejecución'"
%pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# 2. Configuración
from pathlib import Path

WORK = Path("/content")
RAW = WORK / "raw"
DATASET = WORK / "crops"                  # dataset de clasificación: split/especie/recorte.jpg

USE_MAKRAI = True
USE_CLINICAL = True

PAD = 0.30            # margen alrededor de la caja (se ve un poco de agar, como en el servidor)
CROP_SIZE = 160       # lado del recorte guardado (px)
MAX_PER_PLATE = 80    # recortes por placa como máximo (equilibra placas muy pobladas)
MIN_BOX = 12          # cajas más pequeñas (px) se descartan
SPLIT = (0.8, 0.1, 0.1)
SEED = 42

MODEL = "yolo11s-cls.pt"
IMGSZ = 160
EPOCHS = 40
TIME_HOURS = 1.0
BATCH = 256

# código de cada dataset -> especie (tablas 1 de ambos artículos)
SPECIES_MAP = {"makrai:sp01": {"species": "Actinobacillus equuli", "gram": "-", "agar": "blood"}, "makrai:sp02": {"species": "Actinobacillus pleuropneumoniae", "gram": "-", "agar": "chocolate"}, "makrai:sp03": {"species": "Aeromonas hydrophila", "gram": "-", "agar": "blood"}, "makrai:sp04": {"species": "Bacillus cereus", "gram": "+", "agar": "blood"}, "makrai:sp05": {"species": "Bibersteinia trehalosi", "gram": "-", "agar": "blood"}, "makrai:sp06": {"species": "Bordetella bronchiseptica", "gram": "-", "agar": "blood"}, "makrai:sp07": {"species": "Brucella ovis", "gram": "-", "agar": "blood"}, "makrai:sp08": {"species": "Clostridium perfringens", "gram": "+", "agar": "blood"}, "makrai:sp09": {"species": "Corynebacterium pseudotuberculosis", "gram": "+", "agar": "blood"}, "makrai:sp10": {"species": "Erysipelothrix rhusiopathiae", "gram": "+", "agar": "blood"}, "makrai:sp11": {"species": "Escherichia coli", "gram": "-", "agar": "nutrient"}, "makrai:sp12": {"species": "Glaesserella parasuis", "gram": "-", "agar": "chocolate"}, "makrai:sp13": {"species": "Klebsiella pneumoniae", "gram": "-", "agar": "blood, nutrient"}, "makrai:sp14": {"species": "Listeria monocytogenes", "gram": "+", "agar": "blood"}, "makrai:sp15": {"species": "Paenibacillus larvae", "gram": "+", "agar": "blood"}, "makrai:sp16": {"species": "Pasteurella multocida", "gram": "-", "agar": "blood"}, "makrai:sp17": {"species": "Proteus mirabilis", "gram": "-", "agar": "MacConkey"}, "makrai:sp18": {"species": "Pseudomonas aeruginosa", "gram": "-", "agar": "blood"}, "makrai:sp19": {"species": "Rhodococcus equi", "gram": "+", "agar": "blood"}, "makrai:sp20": {"species": "Salmonella enterica", "gram": "-", "agar": "nutrient"}, "makrai:sp21": {"species": "Staphylococcus aureus", "gram": "+", "agar": "blood"}, "makrai:sp22": {"species": "Staphylococcus hyicus", "gram": "+", "agar": "blood"}, "makrai:sp23": {"species": "Streptococcus agalactiae", "gram": "+", "agar": "blood"}, "makrai:sp24": {"species": "Trueperella pyogenes", "gram": "+", "agar": "blood"}, "clinical:aba": {"species": "Acinetobacter baumannii", "gram": "-", "agar": "MacConkey"}, "clinical:eco": {"species": "Escherichia coli", "gram": "-", "agar": "blood"}, "clinical:kpn": {"species": "Klebsiella pneumoniae", "gram": "-", "agar": "MacConkey"}, "clinical:sau": {"species": "Staphylococcus aureus", "gram": "+", "agar": "blood"}, "clinical:ses": {"species": "Salmonella enterica", "gram": "-", "agar": "Salmonella-Shigella"}, "clinical:sma": {"species": "Serratia marcescens", "gram": "-", "agar": "MacConkey"}, "clinical:sep": {"species": "Staphylococcus epidermidis", "gram": "+", "agar": "blood"}, "clinical:efa": {"species": "Enterococcus faecium", "gram": "+", "agar": "blood"}, "clinical:stm": {"species": "Stenotrophomonas maltophilia", "gram": "-", "agar": "MacConkey"}, "clinical:eca": {"species": "Enterococcus casseliflavus", "gram": "+", "agar": "blood"}, "clinical:enc": {"species": "Enterobacter cloacae", "gram": "-", "agar": "MacConkey"}, "clinical:cst": {"species": "Corynebacterium striatum", "gram": "+", "agar": "blood"}, "clinical:mmo": {"species": "Morganella morganii", "gram": "-", "agar": "MacConkey"}, "clinical:bce": {"species": "Bacillus cereus", "gram": "+", "agar": "blood"}, "clinical:spy": {"species": "Streptococcus pyogenes", "gram": "+", "agar": "blood"}, "clinical:sag": {"species": "Streptococcus agalactiae", "gram": "+", "agar": "blood"}, "clinical:ppu": {"species": "Pseudomonas putida", "gram": "-", "agar": "blood"}, "clinical:spn": {"species": "Streptococcus pneumoniae", "gram": "+", "agar": "blood"}, "clinical:bcp": {"species": "Burkholderia cepacia", "gram": "-", "agar": "MacConkey"}}

# --- Datos adicionales (opcional): mismos formatos que en el notebook del detector (COCO o AGAR) -------
# Aquí cada foto necesita su ESPECIE: la categoría de sus anotaciones debe ser el nombre científico
# («Escherichia coli») o traducirse con "species". Con las placas de la app, expórtalas así:
#   python -m bioyologic.manage export-coco --email TU_CORREO --proyecto "E. coli" --especie "Escherichia coli"
# Solo se usan placas con una única especie (en una placa mixta no se sabe cuál es cada colonia).
# Antes de usar un dataset ajeno, revisa su licencia y cítalo en THIRD_PARTY_LICENSES.md.
EXTRA = [
    # {"name": "mi_lab", "path": "/content/drive/MyDrive/bioyologic/ecoli_mi_lab.zip"},
    # {"name": "agar", "path": "/content/drive/MyDrive/agar", "max_plates": 1500,
    #  "species": {"E.coli": "Escherichia coli", "S.aureus": "Staphylococcus aureus"}},
]
# Gram ("+" o "-") y medio de las especies NUEVAS, para mostrarlos en la app (las actuales ya los tienen).
EXTRA_INFO = {
    # "Nombre científico": {"gram": "+", "agar": "blood"},
}

# Clasificador actual de Bioyologic: al final se compara con el nuevo en las MISMAS placas de test.
BASELINE = "https://github.com/bioyologic/bioyologic/raw/main/models/bioyologic_species.pt"   # "" = no comparar

In [ ]:
# 3. Descarga de los datasets
import json, struct, urllib.request, zipfile, zlib
from concurrent.futures import ThreadPoolExecutor

def http_range(url, start, end):
    req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
    return urllib.request.urlopen(req, timeout=300).read()

def remote_zip_entries(url):
    """Índice de un ZIP remoto (también ZIP64) leyendo solo su directorio central."""
    size = int(urllib.request.urlopen(urllib.request.Request(url, method="HEAD")).headers["Content-Length"])
    tail = http_range(url, max(0, size - 65558), size - 1)
    i = tail.rfind(b"PK\x05\x06")
    cd_size, cd_off = struct.unpack("<II", tail[i + 12:i + 20])
    if cd_off == 0xFFFFFFFF or cd_size == 0xFFFFFFFF:
        j = tail.rfind(b"PK\x06\x06")
        cd_size, cd_off = struct.unpack("<QQ", tail[j + 40:j + 56])
    cd = http_range(url, cd_off, cd_off + cd_size - 1)
    entries, p = {}, 0
    while cd[p:p + 4] == b"PK\x01\x02":
        method = struct.unpack("<H", cd[p + 10:p + 12])[0]
        csize, usize = struct.unpack("<II", cd[p + 20:p + 28])
        n, e, c = struct.unpack("<HHH", cd[p + 28:p + 34])
        off = struct.unpack("<I", cd[p + 42:p + 46])[0]
        name = cd[p + 46:p + 46 + n].decode("utf-8", "replace")
        extra, q = cd[p + 46 + n:p + 46 + n + e], 0
        while q + 4 <= len(extra):
            hid, hlen = struct.unpack("<HH", extra[q:q + 4])
            if hid == 1:
                vals, k = extra[q + 4:q + 4 + hlen], 0
                if usize == 0xFFFFFFFF: usize = struct.unpack("<Q", vals[k:k + 8])[0]; k += 8
                if csize == 0xFFFFFFFF: csize = struct.unpack("<Q", vals[k:k + 8])[0]; k += 8
                if off == 0xFFFFFFFF: off = struct.unpack("<Q", vals[k:k + 8])[0]; k += 8
            q += 4 + hlen
        entries[name] = (method, csize, off)
        p += 46 + n + e + c
    return entries

def remote_zip_read(url, entries, name):
    method, csize, off = entries[name]
    n, e = struct.unpack("<HH", http_range(url, off, off + 29)[26:30])
    data = http_range(url, off + 30 + n + e, off + 30 + n + e + csize - 1)
    return data if method == 0 else zlib.decompress(data, -15)

def download(url, dest, size=None, parts=1):
    """Descarga a disco; con parts > 1, en trozos paralelos (Zenodo limita ~1 MB/s por conexión)."""
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    if parts == 1:
        with urllib.request.urlopen(url, timeout=300) as r, open(tmp, "wb") as f:
            while chunk := r.read(8 << 20):
                f.write(chunk)
    else:
        total = int(urllib.request.urlopen(urllib.request.Request(url, method="HEAD")).headers["Content-Length"])
        with open(tmp, "wb") as f:
            f.truncate(total)
        step = -(-total // parts)
        def piece(i):
            start, end = i * step, min(total, (i + 1) * step) - 1
            req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
            with urllib.request.urlopen(req, timeout=300) as r, open(tmp, "r+b") as f:
                f.seek(start)
                while chunk := r.read(8 << 20):
                    f.write(chunk)
        with ThreadPoolExecutor(parts) as pool:
            list(pool.map(piece, range(parts)))
    tmp.rename(dest)

SOURCES = []   # (nombre, coco.json, carpeta de imágenes)

if USE_MAKRAI:
    d = RAW / "makrai"
    d.mkdir(parents=True, exist_ok=True)
    meta = json.load(urllib.request.urlopen("https://api.figshare.com/v2/articles/22022540"))
    files = [f for f in meta["files"] if f["name"].lower().endswith(".jpg") or f["name"] == "annot_COCO.json"]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda f: download(f["download_url"], d / f["name"], f["size"]), files))
    SOURCES.append(("makrai", d / "annot_COCO.json", d))
    print(f"Makrai: {len(files)} archivos")

if USE_CLINICAL:
    d = RAW / "clinical"
    d.mkdir(parents=True, exist_ok=True)
    base = "https://zenodo.org/api/records/18256864/files"
    # Las etiquetas (label.zip, 2,6 GB) incluyen JSON de LabelMe con la imagen embebida:
    # solo se extrae coco.json (~1 MB) leyendo ese fragmento del ZIP remoto.
    if not (d / "coco.json").exists():
        url = f"{base}/label.zip/content"
        (d / "coco.json").write_bytes(remote_zip_read(url, remote_zip_entries(url), "label/coco_json/coco.json"))
    if not (d / "images").exists():
        download(f"{base}/images.zip/content", d / "images.zip", parts=16)
        with zipfile.ZipFile(d / "images.zip") as z:
            z.extractall(d)
        (d / "images.zip").unlink()
    SOURCES.append(("clinical", d / "coco.json", d / "images"))
    print("Clinical: listo")

!du -sh {RAW}/*

In [ ]:
# 3b. Datos adicionales (EXTRA) y modelo actual (BASELINE)
import json, random, urllib.request, zipfile

IMG_EXT = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

def read_json(p):
    try:
        return json.loads(Path(p).read_text(encoding="utf-8"))
    except (ValueError, UnicodeDecodeError):
        return None

def is_coco(d):
    return isinstance(d, dict) and {"images", "annotations", "categories"} <= d.keys()

def agar_to_coco(folder, limit, seed):
    # AGAR trae un .json por foto: {"labels": [{"class", "x", "y", "width", "height"}], "colonies_number": n}.
    # Las placas «incontables» no tienen cajas y se descartan; las vacías (0 colonias) sí valen.
    anns = sorted(Path(folder).rglob("*.json"))
    random.Random(seed).shuffle(anns)
    coco, cats = {"images": [], "annotations": [], "categories": []}, {}
    for ann in anns:
        if limit and len(coco["images"]) >= limit:
            break
        d = read_json(ann)
        img = next((ann.with_suffix(e) for e in IMG_EXT if ann.with_suffix(e).exists()), None)
        if img is None or not isinstance(d, dict) or "labels" not in d:
            continue
        if not d["labels"] and d.get("colonies_number", 0) != 0:
            continue
        iid = len(coco["images"]) + 1
        coco["images"].append({"id": iid, "file_name": img.name})
        for lb in d["labels"]:
            cid = cats.setdefault(str(lb["class"]), len(cats) + 1)
            coco["annotations"].append({"id": len(coco["annotations"]) + 1, "image_id": iid, "category_id": cid,
                                        "bbox": [lb["x"], lb["y"], lb["width"], lb["height"]]})
    coco["categories"] = [{"id": i, "name": n} for n, i in cats.items()]
    return coco

if any(str(e["path"]).startswith("/content/drive") for e in EXTRA):
    from google.colab import drive
    drive.mount("/content/drive")

EXTRA_SOURCES = []   # (nombre, coco.json, carpeta de imágenes, máximo de placas)
for e in EXTRA:
    name, src = e["name"], Path(e["path"])
    assert name not in ("base", "makrai", "clinical") and name not in [x[0] for x in EXTRA_SOURCES], \
        f"Nombre no válido o repetido en EXTRA: {name}"
    assert src.exists(), f"EXTRA «{name}»: no existe {src}"
    folder = src
    if src.suffix.lower() == ".zip":
        folder = RAW / "extra" / name
        if not folder.exists():
            with zipfile.ZipFile(src) as z:
                z.extractall(folder)
    coco_path = next((p for p in sorted(folder.rglob("*.json"))[:20] if is_coco(read_json(p))), None)
    kind = "COCO"
    if coco_path is None:
        kind = "AGAR"
        coco = agar_to_coco(folder, e.get("max_plates"), SEED)
        assert coco["images"], f"EXTRA «{name}»: en {src} no hay fotos anotadas en formato COCO ni AGAR"
        coco_path = RAW / "extra" / f"{name}_coco.json"
        coco_path.parent.mkdir(parents=True, exist_ok=True)
        coco_path.write_text(json.dumps(coco), encoding="utf-8")
    EXTRA_SOURCES.append((name, coco_path, folder, e.get("max_plates")))
    print(f"EXTRA «{name}»: formato {kind}, {len(read_json(coco_path)['images'])} fotos anotadas")
if not EXTRA:
    print("Sin datos adicionales: se entrena solo con los datasets base.")

BASELINE_PT = None
if BASELINE:
    BASELINE_PT = Path(BASELINE) if Path(BASELINE).exists() else WORK / ("actual_" + BASELINE.split("/")[-1])
    if not BASELINE_PT.exists():
        urllib.request.urlretrieve(BASELINE, BASELINE_PT)
    print("Modelo actual para comparar:", BASELINE_PT.name, f"({BASELINE_PT.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# 4. Recortes de colonias por especie (partición por placa dentro de cada especie)
import json, random, shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import cv2
import numpy as np

def square_crop(img, box, pad=PAD):
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * (1 + 2 * pad)
    h, w = img.shape[:2]
    a, b = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    c, d = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    crop = img[b:d, a:c]
    return cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA) if crop.size else None

def load_plates(coco_path, img_dir, species_of, pure_only=False):
    # especie -> [(ruta, [cajas])]. La especie de la placa es la de la mayoría de sus colonias.
    coco = json.loads(Path(coco_path).read_text(encoding="utf-8"))
    cats = {c["id"]: species_of(c["name"]) for c in coco["categories"]}
    index = {p.name: p for p in Path(img_dir).rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    by_img = defaultdict(list)
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        if max(w, h) >= MIN_BOX:
            by_img[a["image_id"]].append((cats[a["category_id"]], (x, y, x + w, y + h)))
    out, mixed = defaultdict(list), 0
    for im in coco["images"]:
        path = index.get(Path(im["file_name"]).name)
        boxes = by_img.get(im["id"], [])
        if path is None or not boxes:
            continue
        kinds = sorted(set(b[0] for b in boxes))
        if pure_only and len(kinds) > 1:
            mixed += 1
            continue
        sp = max(kinds, key=lambda s: sum(1 for b in boxes if b[0] == s))
        out[sp].append((path, [b[1] for b in boxes]))
    if mixed:
        print(f"  {mixed} placas mixtas descartadas")
    return out

def split_species(found, rng):
    out = {"train": [], "val": [], "test": []}
    for sp, items in sorted(found.items()):
        rng.shuffle(items)
        n = len(items)
        n_te = max(1, round(n * SPLIT[2])) if n >= 3 else 0
        n_va = max(1, round(n * SPLIT[1])) if n >= 3 else 0
        out["test"] += [(sp, it) for it in items[:n_te]]
        out["val"] += [(sp, it) for it in items[n_te:n_te + n_va]]
        out["train"] += [(sp, it) for it in items[n_te + n_va:]]
    return out

plates = defaultdict(list)   # especie -> [(ruta, [cajas])]
for src_name, coco_path, img_dir in SOURCES:
    found = load_plates(coco_path, img_dir, lambda cat, src=src_name: SPECIES_MAP[f"{src}:{cat}"]["species"])
    for sp, items in found.items():
        plates[sp] += items
# Los datasets base se reparten SIEMPRE igual (misma semilla, sin mezclarlos con los datos adicionales):
# su test son las mismas placas con las que se midió el clasificador actual, que nunca las vio al entrenar.
parts = split_species(plates, random.Random(SEED))
TEST_BASE, TEST_EXTRA, PREFIX = list(parts["test"]), {}, {}
KNOWN = {v["species"] for v in SPECIES_MAP.values()}
for src_name, coco_path, img_dir, limit in EXTRA_SOURCES:
    table = next(e for e in EXTRA if e["name"] == src_name).get("species", {})
    def species_of(cat, table=table, src=src_name):
        sp = table.get(cat, cat)
        assert sp.lower() != "colony", (f"EXTRA «{src}» no dice la especie de sus fotos: expórtalas con --especie "
                                        'o añade "species": {"colony": "Nombre científico"} a su entrada')
        return sp
    found = load_plates(coco_path, img_dir, species_of, pure_only=True)
    if limit:   # como mucho `limit` placas, al azar
        flat = [(sp, it) for sp, items in sorted(found.items()) for it in items]
        random.Random(SEED).shuffle(flat)
        found = defaultdict(list)
        for sp, it in flat[:limit]:
            found[sp].append(it)
    for sp, items in sorted(found.items()):
        if sp not in KNOWN:
            print(f"⚠️ «{src_name}»: especie nueva «{sp}» ({len(items)} placas). Si es otro nombre de una especie "
                  'que ya existe, tradúcelo con "species".')
        assert sp in plates or len(items) >= 3, \
            f"«{src_name}»: de una especie nueva ({sp}) hacen falta al menos 3 placas"
    extra = split_species(found, random.Random(SEED))
    for part in parts:
        parts[part] = parts[part] + extra[part]
    TEST_EXTRA[src_name] = extra["test"]
    PREFIX.update({it[0]: src_name for items in found.values() for it in items})
    print(f"EXTRA «{src_name}»: {sum(len(v) for v in found.values())} placas de {len(found)} especies")

shutil.rmtree(DATASET, ignore_errors=True)

def export(args):
    part, sp, (path, boxes) = args
    img = cv2.imread(str(path))
    if img is None:
        return 0
    if len(boxes) > MAX_PER_PLATE:
        boxes = random.Random(str(path)).sample(boxes, MAX_PER_PLATE)
    folder = DATASET / part / sp.replace(" ", "_")
    folder.mkdir(parents=True, exist_ok=True)
    # dos fuentes adicionales pueden traer archivos con el mismo nombre
    stem = f"{PREFIX[path]}_{path.stem}" if path in PREFIX else path.stem
    n = 0
    for k, b in enumerate(boxes):
        crop = square_crop(img, b)
        if crop is not None:
            cv2.imwrite(str(folder / f"{stem}_{k}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 92])
            n += 1
    return n

summary = {}
for part, items in parts.items():
    with ThreadPoolExecutor(4) as pool:
        counts = list(pool.map(export, [(part, sp, it) for sp, it in items]))
    summary[part] = {"placas": len(items), "recortes": sum(counts)}
TEST_PLATES = parts["test"]
CLASSES = sorted({sp for items in parts.values() for sp, _ in items})
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Placas de test:", {"base": len(TEST_BASE), **{k: len(v) for k, v in TEST_EXTRA.items()}})
print(len(CLASSES), "especies:", ", ".join(CLASSES))

In [ ]:
# 5. Un recorte de ejemplo por especie
import matplotlib.pyplot as plt
cols = 8
rows = -(-len(CLASSES) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.5 * rows))
for ax in axes.ravel():
    ax.axis("off")
for ax, sp in zip(axes.ravel(), CLASSES):
    files = sorted((DATASET / "train" / sp.replace(" ", "_")).glob("*.jpg"))
    if files:
        ax.imshow(cv2.cvtColor(cv2.imread(str(files[len(files) // 2])), cv2.COLOR_BGR2RGB))
    ax.set_title(sp, fontsize=8, style="italic")
plt.tight_layout(); plt.show()

In [ ]:
# 6. Entrenamiento del clasificador
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(data=str(DATASET), epochs=EPOCHS, time=TIME_HOURS, imgsz=IMGSZ, batch=BATCH, patience=10,
            workers=2, project=str(WORK / "runs"), name="species", exist_ok=True, cos_lr=True,
            fliplr=0.5, flipud=0.5, degrees=180, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3)
BEST = Path(model.trainer.best)
m = YOLO(str(BEST)).val(data=str(DATASET), split="test", imgsz=IMGSZ)
print(f"TEST por colonia: top-1 = {m.top1:.3f} · top-5 = {m.top5:.3f}")

In [ ]:
# 7. Precisión por PLACA: votación entre las colonias de cada placa de test
from collections import Counter

def evaluate_species(weights, test_plates):
    clf = YOLO(str(weights))
    labels = {i: n.replace("_", " ") for i, n in clf.names.items()}
    rows, ok_cols, n_cols = [], 0, 0
    for sp, (path, boxes) in test_plates:
        img = cv2.imread(str(path))
        if img is None:
            continue
        crops = [c for c in (square_crop(img, b) for b in boxes[:MAX_PER_PLATE]) if c is not None]
        if not crops:
            continue
        probs = np.zeros(len(labels))
        for i in range(0, len(crops), 128):
            for r in clf.predict(crops[i:i + 128], imgsz=IMGSZ, verbose=False):
                p = r.probs.data.cpu().numpy()
                probs += p
                ok_cols += int(labels[int(p.argmax())] == sp)
        n_cols += len(crops)
        rows.append((sp, labels[int(probs.argmax())], float(probs.max()) / len(crops)))
    return {"placas": len(rows), "placas_ok": sum(t == p for t, p, _ in rows), "colonias": n_cols,
            "top1": round(ok_cols / max(1, n_cols), 4), "rows": rows}

names = YOLO(str(BEST)).names
res = evaluate_species(BEST, TEST_PLATES)
rows, ok = res["rows"], res["placas_ok"]
print(f"Placas de test: {len(rows)} · especie correcta en {ok} ({100 * ok / max(1, len(rows)):.1f} %)")
print(f"Acierto por colonia en esas placas: {100 * res['top1']:.1f} %")
errors = Counter((t, p) for t, p, _ in rows if t != p)
for (t, p), n in errors.most_common(10):
    print(f"  {t} → confundida con {p} ({n} placa{'s' if n > 1 else ''})")

In [ ]:
# 8. ¿Conviene reemplazar el clasificador actual? Los dos se miden sobre las mismas placas de test
TOL_TOP1 = 0.005   # acierto por colonia que el nuevo puede perder en las placas base (ruido entre entrenamientos)

COMPARISON, VERDICT = {}, None
if BASELINE_PT is None:
    print("No se compara: BASELINE está vacío.")
else:
    for subset, test_plates in {"base": TEST_BASE, **TEST_EXTRA}.items():
        if test_plates:
            COMPARISON[subset] = {who: {k: v for k, v in evaluate_species(w, test_plates).items() if k != "rows"}
                                  for who, w in (("actual", BASELINE_PT), ("nuevo", BEST))}
    print(f"{'Test':<14}{'Modelo':<8}{'Placas':>7}{'Correctas':>10}{'Colonias':>9}{'Por colonia':>12}")
    for subset, r in COMPARISON.items():
        for who in ("actual", "nuevo"):
            m = r[who]
            print(f"{subset:<14}{who:<8}{m['placas']:>7}{m['placas_ok']:>10}{m['colonias']:>9}{100 * m['top1']:>10.1f} %")

    # Regla: el nuevo no puede acertar menos placas ni menos colonias en las placas base y, si hay datos nuevos,
    # tiene que identificarlos mejor. Sin datos nuevos, tiene que mejorar el acierto por colonia en las base.
    reasons, notes = [], []
    base = COMPARISON.get("base")
    extras = {k: v for k, v in COMPARISON.items() if k != "base"}
    if not COMPARISON:
        reasons.append("no hay placas de test con las que comparar")
    if base:
        a, n = base["actual"], base["nuevo"]
        if n["placas_ok"] < a["placas_ok"]:
            reasons.append(f"acierta menos placas base ({a['placas_ok']} → {n['placas_ok']} de {n['placas']})")
        if n["top1"] < a["top1"] - TOL_TOP1:
            reasons.append(f"acierta menos colonias en las placas base ({100 * a['top1']:.1f} % → {100 * n['top1']:.1f} %)")
        if not extras and n["top1"] <= a["top1"]:
            reasons.append("sin datos nuevos, tampoco mejora el acierto por colonia en las placas base")
    for subset, r in extras.items():
        a, n = r["actual"], r["nuevo"]
        if (n["placas_ok"], n["top1"]) <= (a["placas_ok"], a["top1"]):
            reasons.append(f"no identifica mejor «{subset}» (placas {a['placas_ok']} → {n['placas_ok']} de {n['placas']}, "
                           f"colonias {100 * a['top1']:.1f} % → {100 * n['top1']:.1f} %)")
        if n["placas"] < 10:
            notes.append(f"«{subset}» solo tiene {n['placas']} placa(s) de test: su resultado es poco fiable")
    for src in EXTRA_SOURCES:
        if not TEST_EXTRA.get(src[0]):
            notes.append(f"«{src[0]}» tiene tan pocas placas por especie que ninguna quedó para test")
    VERDICT = "conservar" if reasons else "reemplazar"
    print()
    if VERDICT == "reemplazar":
        print("✅ El clasificador NUEVO es mejor: se puede reemplazar el actual.")
    else:
        print("❌ CONSERVA el clasificador actual. El nuevo:")
        for r in reasons:
            print("   ·", r)
    for n in notes:
        print("⚠️", n)
    (WORK / "comparacion_especies.json").write_text(json.dumps(
        {"veredicto": VERDICT, "motivos": reasons, "avisos": notes, "resultados": COMPARISON},
        indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
# 9. Exportar para Bioyologic
out = WORK / "bioyologic_species.pt"
shutil.copy(BEST, out)
info = {v["species"]: {"gram": v["gram"], "agar": v["agar"]} for v in SPECIES_MAP.values()}
for sp in CLASSES:   # especies nuevas que llegaron con los datos adicionales
    if sp not in info:
        info[sp] = EXTRA_INFO.get(sp, {"gram": "", "agar": ""})
        if sp not in EXTRA_INFO:
            print(f"⚠️ Falta el Gram y el medio de «{sp}» en EXTRA_INFO: la app los mostrará vacíos.")
(WORK / "bioyologic_species.json").write_text(json.dumps({
    "pad": PAD, "imgsz": IMGSZ, "classes": [names[i].replace("_", " ") for i in range(len(names))],
    "species_info": info, "datasets": [s[0] for s in SOURCES] + [s[0] for s in EXTRA_SOURCES],
}, indent=1, ensure_ascii=False))
# Un solo archivo: Chrome bloquea las descargas múltiples automáticas.
import zipfile
bundle = WORK / "bioyologic_species.zip"
with zipfile.ZipFile(bundle, "w") as z:
    z.write(out, out.name)
    z.write(WORK / "bioyologic_species.json", "bioyologic_species.json")
    if (WORK / "comparacion_especies.json").exists():
        z.write(WORK / "comparacion_especies.json", "comparacion_especies.json")
if VERDICT == "conservar":
    print("⚠️ Este clasificador NO superó al actual (celda 8): no lo copies a models/. Se descarga solo para revisarlo.")
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print("Copia manualmente:", bundle)

## Siguiente paso
**Solo si la celda 8 dice que el clasificador nuevo es mejor** (o si no había uno actual con el que comparar):

1. Copia `bioyologic_species.pt` y `bioyologic_species.json` en `Bioyologic/models/`, sustituyendo los anteriores.
2. Ejecuta `python backend/scripts/export_onnx.py` (el servidor usa la versión ONNX) y las pruebas del backend.
3. Si añadiste **especies nuevas**, agrégalas a la biblioteca de la app (`backend/bioyologic/library.py`): una prueba comprueba que cada especie del clasificador tiene su ficha.
4. Actualiza las cifras de `MODEL_CARD.md` y del `README.md` con las de `comparacion_especies.json`, y añade los datos nuevos a `THIRD_PARTY_LICENSES.md` si son de terceros.
5. Sube el cambio y actualiza el servidor.

Si la celda 8 dice que conserves el clasificador actual, no cambies nada.